# Sandbox

In [1]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [30]:
import ocha_stratus as stratus
import pandas as pd

from src.utils.timeseries import detrend_column
from src.utils.rp_calc import calculate_one_group_rp

In [3]:
query = """
SELECT *
FROM public.seas5
WHERE pcode = 'SO'
"""

In [7]:
df = pd.read_sql(
    query,
    stratus.get_engine(stage="prod"),
    parse_dates=["valid_date", "issued_date"],
)

In [5]:
df

,iso3,pcode,valid_date,issued_date,leadtime,adm_level,mean,median,min,max,count,sum,std
0,SOM,SO,1991-08-01,1991-08-01,0,0,0.408863,0.213034,0.013126,3.306107,20676,8453.6590,0.521375
1,SOM,SO,1991-08-01,1991-07-01,1,0,0.329535,0.189156,0.014192,2.112049,20676,6813.4590,0.335994
2,SOM,SO,1991-08-01,1991-06-01,2,0,0.338107,0.178797,0.002777,2.226891,20676,6990.7080,0.387419
3,SOM,SO,1991-08-01,1991-05-01,3,0,0.304106,0.156325,0.004948,2.195641,20676,6287.6973,0.352408
4,SOM,SO,1991-08-01,1991-04-01,4,0,0.292625,0.141966,0.003313,2.324412,20676,6050.3047,0.358042
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3747,SOM,SO,2025-10-01,2025-08-01,2,0,1.361580,1.261131,0.256190,3.577864,20676,28152.0210,0.737426
3748,SOM,SO,2025-11-01,2025-08-01,3,0,1.161147,0.813266,0.167700,4.682526,20676,24007.8830,0.884882
3749,SOM,SO,2025-12-01,2025-08-01,4,0,0.412079,0.251531,0.039641,2.819808,20676,8520.1480,0.404910
3750,SOM,SO,2026-01-01,2025-08-01,5,0,0.090552,0.044498,0.003562,1.696177,20676,1872.2588,0.130859


In [37]:
issued_month = 8
# valid_months = [10, 11, 12]
valid_months = [11, 12, 1]

In [38]:
df_issue = (
    df[
        df["valid_date"].dt.month.isin(valid_months)
        & (df["issued_date"].dt.month == issued_month)
    ]
    .groupby(df["issued_date"].dt.year)["mean"]
    .mean()
    .rename_axis("year")
    .reset_index()
)

In [39]:
df_issue = detrend_column(df_issue, col="mean", index_col="year")

In [40]:
df_issue

,year,mean,mean_detrended
0,1981,0.641801,0.648619
1,1982,0.950034,0.956542
2,1983,0.586856,0.593054
3,1984,0.580985,0.586874
4,1985,0.582941,0.588519
5,1986,0.712863,0.718131
6,1987,0.764532,0.769491
7,1988,0.486425,0.491074
8,1989,0.608558,0.612897
9,1990,0.655173,0.659202


In [41]:
df_issue = calculate_one_group_rp(df_issue, col_name="mean", ascending=True)
df_issue = calculate_one_group_rp(
    df_issue, col_name="mean_detrended", ascending=True
)

In [42]:
df_issue.sort_values("mean_rank")

,year,mean,mean_detrended,mean_rank,mean_rp,mean_detrended_rank,mean_detrended_rp
41,2022,0.454499,0.448611,1.0,46.000000,1.0,46.000000
7,1988,0.486425,0.491074,2.0,23.000000,3.0,15.333333
27,2008,0.488986,0.487437,3.0,15.333333,2.0,23.000000
17,1998,0.492238,0.493788,4.0,11.500000,5.0,9.200000
32,2013,0.494533,0.491434,5.0,9.200000,4.0,11.500000
18,1999,0.504121,0.505361,6.0,7.666667,6.0,7.666667
19,2000,0.524296,0.525226,7.0,6.571429,8.0,5.750000
40,2021,0.527095,0.521516,8.0,5.750000,7.0,6.571429
35,2016,0.531912,0.527883,9.0,5.111111,9.0,5.111111
15,1996,0.537787,0.539957,10.0,4.600000,10.0,4.600000
